
# Phase 2: Forecasting Target Construction

## Objective

This phase constructs future energy-consumption targets for two forecasting horizons:

- **15-minute-ahead forecasting:** predict energy consumption at the next observation.
- **1-hour-ahead forecasting:** predict energy consumption four observations ahead.

The dataset is recorded at 15-minute intervals. The data will be sorted chronologically and validated before target construction.

**Leakage prevention:** Future consumption values are used only to create the target variables. They must not be included as predictor features.


In [1]:

import pandas as pd
from pathlib import Path

# Define input and output paths
INPUT_PATH = Path("../Data/processed/steel_industry_clean.csv")
OUTPUT_PATH = Path("../Data/processed/steel_industry_phase2_targets.csv")

# Load the cleaned dataset
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {INPUT_PATH.resolve()}"
    )

df_phase2 = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])

# Validate required columns
required_columns = ["timestamp", "Usage_kWh"]
missing_columns = [
    col for col in required_columns
    if col not in df_phase2.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Validate timestamp and target values
if df_phase2["timestamp"].isna().any():
    raise ValueError("Missing or invalid timestamps found.")

df_phase2["Usage_kWh"] = pd.to_numeric(
    df_phase2["Usage_kWh"], errors="coerce"
)

if df_phase2["Usage_kWh"].isna().any():
    raise ValueError("Missing or non-numeric Usage_kWh values found.")

# Sort chronologically
df_phase2 = (
    df_phase2
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Dataset loaded successfully.")
print("Rows:", f"{len(df_phase2):,}")
print("Columns:", df_phase2.shape[1])
print("First timestamp:", df_phase2["timestamp"].min())
print("Last timestamp:", df_phase2["timestamp"].max())

display(df_phase2.head())


Dataset loaded successfully.
Rows: 35,040
Columns: 14
First timestamp: 2018-01-01 00:00:00
Last timestamp: 2018-12-31 23:45:00


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,timestamp,zero_usage_flag,high_usage_flag
0,01-01-2018 00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load,2018-01-01 00:00:00,False,False
1,01-01-2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load,2018-01-01 00:15:00,False,False
2,01-01-2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load,2018-01-01 00:30:00,False,False
3,01-01-2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load,2018-01-01 00:45:00,False,False
4,01-01-2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load,2018-01-01 01:00:00,False,False



## 1. Validate the Time Series

Target creation using row shifts assumes that observations are recorded at regular 15-minute intervals.

Before creating targets, verify that timestamps are unique and consecutive observations are separated by exactly 15 minutes. If intervals are irregular, stop and investigate rather than silently treating the next row as the next 15-minute period.


In [2]:

expected_interval = pd.Timedelta(minutes=15)

# Check timestamp uniqueness
if df_phase2["timestamp"].duplicated().any():
    raise ValueError("Duplicate timestamps found.")

# Check consecutive intervals
time_intervals = df_phase2["timestamp"].diff().dropna()

incorrect_intervals = time_intervals[
    time_intervals != expected_interval
]

print("Timestamps unique:", df_phase2["timestamp"].is_unique)
print("Chronologically sorted:", df_phase2["timestamp"].is_monotonic_increasing)
print("Expected interval:", expected_interval)
print("Unexpected intervals:", len(incorrect_intervals))

if not incorrect_intervals.empty:
    display(incorrect_intervals.head(10).to_frame("Observed interval"))
    raise ValueError(
        "Irregular time intervals found. Investigate before creating targets."
    )

print("Time-series validation passed.")


Timestamps unique: True
Chronologically sorted: True
Expected interval: 0 days 00:15:00
Unexpected intervals: 0
Time-series validation passed.



## 2. Construct Forecasting Targets

Let \(Y_t\) denote energy consumption at time \(t\).

The two targets are defined as:

- **15-minute horizon:** \(Y_{t+1}\)
- **1-hour horizon:** \(Y_{t+4}\)

Because the sampling interval is 15 minutes, shifting the consumption column by -1 or -4 aligns each observation with its future target.

The final observation has no 15-minute target, and the final four observations have no one-hour target. These missing values are expected and should not be filled with artificial values.


In [3]:

# Create future energy-consumption targets
df_phase2["target_15min"] = df_phase2["Usage_kWh"].shift(-1)
df_phase2["target_1hour"] = df_phase2["Usage_kWh"].shift(-4)

# Inspect target alignment
display(
    df_phase2[
        ["timestamp", "Usage_kWh", "target_15min", "target_1hour"]
    ].head(10)
)

print("15-minute target missing values:",
      df_phase2["target_15min"].isna().sum())

print("1-hour target missing values:",
      df_phase2["target_1hour"].isna().sum())


,timestamp,Usage_kWh,target_15min,target_1hour
0,2018-01-01 00:00:00,3.42,3.17,3.31
1,2018-01-01 00:15:00,3.17,4.00,3.82
2,2018-01-01 00:30:00,4.00,3.24,3.28
3,2018-01-01 00:45:00,3.24,3.31,3.60
4,2018-01-01 01:00:00,3.31,3.82,3.60
5,2018-01-01 01:15:00,3.82,3.28,3.28
6,2018-01-01 01:30:00,3.28,3.60,3.78
7,2018-01-01 01:45:00,3.60,3.60,3.46
8,2018-01-01 02:00:00,3.60,3.28,3.24
9,2018-01-01 02:15:00,3.28,3.78,3.96


15-minute target missing values: 1
1-hour target missing values: 4



## 3. Validate Target Alignment

Verify that the generated target columns correspond to the correct future observations. Also check the expected number of missing targets at the end of the time series.

These checks help prevent forecasting at the wrong horizon due to incorrect shifts or data ordering.


In [4]:

# Verify 15-minute and one-hour target alignment
assert (
    df_phase2["target_15min"].iloc[:-1].reset_index(drop=True)
    .equals(
        df_phase2["Usage_kWh"].shift(-1).iloc[:-1].reset_index(drop=True)
    )
)

assert (
    df_phase2["target_1hour"].iloc[:-4].reset_index(drop=True)
    .equals(
        df_phase2["Usage_kWh"].shift(-4).iloc[:-4].reset_index(drop=True)
    )
)

# Verify expected missing values at the end
assert df_phase2["target_15min"].isna().sum() == 1
assert df_phase2["target_1hour"].isna().sum() == 4

print("Target alignment checks passed.")
print("15-minute target missing values:", df_phase2["target_15min"].isna().sum())
print("1-hour target missing values:", df_phase2["target_1hour"].isna().sum())


Target alignment checks passed.
15-minute target missing values: 1
1-hour target missing values: 4



## 4. Export the Target-Engineered Dataset

Save the dataset with both future target columns for use in the subsequent feature-engineering and forecasting phases.

The final rows with unavailable future targets are retained in this intermediate dataset. They can be excluded when preparing training data for each forecasting horizon.


In [5]:

# Ensure output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the target-engineered dataset
df_phase2.to_csv(OUTPUT_PATH, index=False)

print("Phase 2 completed successfully.")
print("Output shape:", df_phase2.shape)
print("Saved to:", OUTPUT_PATH.resolve())


Phase 2 completed successfully.
Output shape: (35040, 16)
Saved to: D:\Research Paper Codes\Data\processed\steel_industry_phase2_targets.csv
